In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor,AdaBoostRegressor
from sklearn.svm import SVR
from sklearn.metrics import mean_absolute_error,mean_squared_error,r2_score
from sklearn.model_selection import GridSearchCV,RandomizedSearchCV
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
import warnings

In [3]:
df=pd.read_csv('../data/StudentsPerformance.csv')

In [4]:
df.head()

,gender,race/ethnicity,parental level of education,lunch,test preparation course,math score,reading score,writing score
0,female,group B,bachelor's degree,standard,none,72,72,74
1,female,group C,some college,standard,completed,69,90,88
2,female,group B,master's degree,standard,none,90,95,93
3,male,group A,associate's degree,free/reduced,none,47,57,44
4,male,group C,some college,standard,none,76,78,75


In [5]:
X=df.drop('math score',axis=1)

In [6]:
Y=df['math score']

In [7]:
X

,gender,race/ethnicity,parental level of education,lunch,test preparation course,reading score,writing score
0,female,group B,bachelor's degree,standard,none,72,74
1,female,group C,some college,standard,completed,90,88
2,female,group B,master's degree,standard,none,95,93
3,male,group A,associate's degree,free/reduced,none,57,44
4,male,group C,some college,standard,none,78,75
...,...,...,...,...,...,...,...
995,female,group E,master's degree,standard,completed,99,95
996,male,group C,high school,free/reduced,none,55,55
997,female,group C,high school,free/reduced,completed,71,65
998,female,group D,some college,standard,completed,78,77


In [8]:
Y

0      72
1      69
2      90
3      47
4      76
       ..
995    88
996    62
997    59
998    68
999    77
Name: math score, Length: 1000, dtype: int64

In [9]:
print("categories in gender:", df['gender'].unique())
print("categories in race/ethnicity:", df['race/ethnicity'].unique())
print("categories in parental level of education:", df['parental level of education'].unique())
print("categories in lunch:", df['lunch'].unique())
print("categories in test preparation course:", df['test preparation course'].unique())
 

categories in gender: ['female' 'male']
categories in race/ethnicity: ['group B' 'group C' 'group A' 'group D' 'group E']
categories in parental level of education: ["bachelor's degree" 'some college' "master's degree" "associate's degree"
 'high school' 'some high school']
categories in lunch: ['standard' 'free/reduced']
categories in test preparation course: ['none' 'completed']


In [10]:
num_features=X.select_dtypes(exclude="object").columns
cat_features=X.select_dtypes(include="object").columns
from sklearn.preprocessing import StandardScaler,OneHotEncoder
from sklearn.compose import ColumnTransformer

numeric_transformer=StandardScaler()
categorical_transformer=OneHotEncoder()
preprocessor=ColumnTransformer(
    transformers=[
        ('num',numeric_transformer,num_features),
        ('cat',categorical_transformer,cat_features)
    ])


In [11]:
X=preprocessor.fit_transform(X)
X

array([[ 0.19399858,  0.39149181,  1.        , ...,  1.        ,
         0.        ,  1.        ],
       [ 1.42747598,  1.31326868,  1.        , ...,  1.        ,
         1.        ,  0.        ],
       [ 1.77010859,  1.64247471,  1.        , ...,  1.        ,
         0.        ,  1.        ],
       ...,
       [ 0.12547206, -0.20107904,  1.        , ...,  0.        ,
         1.        ,  0.        ],
       [ 0.60515772,  0.58901542,  1.        , ...,  1.        ,
         1.        ,  0.        ],
       [ 1.15336989,  1.18158627,  1.        , ...,  0.        ,
         0.        ,  1.        ]], shape=(1000, 19))

In [12]:
X_train,X_test,Y_train,Y_test=train_test_split(X,Y,test_size=0.2,random_state=42)

In [13]:
X_train

array([[ 0.05694554,  0.45733301,  1.        , ...,  1.        ,
         0.        ,  1.        ],
       [ 0.94779033,  0.98406266,  1.        , ...,  0.        ,
         1.        ,  0.        ],
       [ 1.35894946,  1.18158627,  1.        , ...,  0.        ,
         0.        ,  1.        ],
       ...,
       [-0.49126664, -0.99117351,  1.        , ...,  1.        ,
         0.        ,  1.        ],
       [-1.45063795, -0.99117351,  0.        , ...,  0.        ,
         1.        ,  0.        ],
       [ 1.4960025 ,  1.37910989,  1.        , ...,  1.        ,
         0.        ,  1.        ]], shape=(800, 19))

In [14]:
def evaluate_model(true,pred):
    mae=mean_absolute_error(true,pred)
    mse=mean_squared_error(true,pred)
    r2=r2_score(true,pred)
    print(f"MAE: {mae}")
    print(f"MSE: {mse}")
    print(f"R2 Score: {r2}")
    return mae,mse,r2

In [15]:
from sklearn.linear_model import Lasso,Ridge,ElasticNet

In [16]:
from sklearn.neighbors import KNeighborsRegressor

In [17]:
models={
      "linear": LinearRegression(),
    "decision_tree": DecisionTreeRegressor(),
    "random_forest": RandomForestRegressor(),
    "ada_boost": AdaBoostRegressor(),
    "svr": SVR(),
    "lasso": Lasso(),
    "ridge": Ridge(),
    "elasticnet": ElasticNet(),
    "knn": KNeighborsRegressor(),
    "xgb": XGBRegressor(),
    "catboost": CatBoostRegressor(verbose=0)
}
model_list=[]
for i in range(len(list(models))):
    model=list(models.values())[i]
    model.fit(X_train,Y_train)
    Y_train_pred=model.predict(X_train)
    Y_test_pred=model.predict(X_test)

    model_train_mae,model_train_rmse,model_train_r2=evaluate_model(Y_train,Y_train_pred)
    model_test_mae,model_test_rmse,model_test_r2=evaluate_model(Y_test,Y_test_pred)

    print(list(models.keys())[i])
    model_list.append(list(models.keys())[i])
    print(f"Train MAE: {model_train_mae}, Train RMSE: {model_train_rmse}, Train R2: {model_train_r2}")
    print(f"Test MAE: {model_test_mae}, Test RMSE: {model_test_rmse}, Test R2: {model_test_r2}")
  


MAE: 4.266711846071956
MSE: 28.33487038064859
R2 Score: 0.8743172040139593
MAE: 4.214763142474851
MSE: 29.095169866715477
R2 Score: 0.8804332983749565
linear
Train MAE: 4.266711846071956, Train RMSE: 28.33487038064859, Train R2: 0.8743172040139593
Test MAE: 4.214763142474851, Test RMSE: 29.095169866715477, Test R2: 0.8804332983749565
MAE: 0.01875
MSE: 0.078125
R2 Score: 0.9996534669718089
MAE: 6.175
MSE: 59.725
R2 Score: 0.7545599050540317
decision_tree
Train MAE: 0.01875, Train RMSE: 0.078125, Train R2: 0.9996534669718089
Test MAE: 6.175, Test RMSE: 59.725, Test R2: 0.7545599050540317
MAE: 1.8160861607142857
MSE: 5.15584372404691
R2 Score: 0.9771306222262498
MAE: 4.6212875
MSE: 35.82823617013889
R2 Score: 0.8527637390147265
random_forest
Train MAE: 1.8160861607142857, Train RMSE: 5.15584372404691, Train R2: 0.9771306222262498
Test MAE: 4.6212875, Test RMSE: 35.82823617013889, Test R2: 0.8527637390147265
MAE: 4.7081799982833
MSE: 33.15162957027743
R2 Score: 0.852951877318925
MAE: 4.755